## Make ras2fim/ripple fim 100 boundary dataset

#### Notes
- MC means model_collection, also known as a FC, folder collections. Each rtx MC represents a root folder from RTX such as mip_01010002, or ble_12090301.
- The final product is three files (names are configurable)
    - A metrics file: Include download load times, upload times and other metrics about process of donwloading the data. File name for now is fim_100_metrics.csv
      This file is designed to be dropped into our google sheet that help do some totals. 
    - A hecras_boundaries file: A file used by HydroVIS that tells us which huc8s have ras2fim, ripple ble or ripple mip data. It includes huc geometry from WBD.
      Comes out in 5070.
    - A hecras_hucs file: This is a simplified version for HydroVIS wihtuot geometry and has less columns. They use while processign feature to help give know
      to look for features in ras2fim, ripple ble, ripple mip or default to hand. Yes.. no features here.

### Finer detail notes 
-  For FIM100 has 992 raw mc's, 980 valid MC's, and 12 ras2fim v2 HUC folders. 
-  All RTX folders have the convention of {source}_{huc8}{sometimes is has an underscore and a name}. ie) mip_01010002, ble_12090301, ble_08040102_UpperOuachita, etc
-  When the set of mc downloads where calculated, a master download list was used which filtered out some bad recs. 
        Some with invalid HUC or HUC lengths, some other bad data.That list named rtx_fim100_valid_mc_folder_names.txt was split into smaller sets
        of file and feed into the FIM repo, get_s3_folders_from_list.sh.
        
### This comes in mulitple processing steps.
- Part 1 : Aggregate stats csvs (has one per download)
- Part 2 : Load orig RTX list, merge with agg stats df, adjust df for data type, bad data, etc
- Part 3 : Merge ras2fim v2 data into it. It has 12 datasets based in HUCs not MC
- Part 4 : Pivot to become HUC centric
- Part 5 : Make a HydroVIS table called hecras_boundaries for the static services, inc add geometries from wbd
- Part 6 : Make a HydroVIS csv called hecras_hucs for flow processing ras2fim and ripple (no geometry, less columns)

Last edited: Jul 11, 2025 for RTX Fim 100


In [ ]:
# GLOBAL ROOT PATH VARIABLE

# NOTE: Careful about checking some of this in if it has actual server names or paths

# ROOT_PATH = "{/our efs root}"
ROOT_PATH = "/efs-drives/fim-dev-efs/fim-data"


In [ ]:
import os
import glob
import stat

import geopandas as gpd
import numpy as np
import pandas as pd

# Display all rows
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_rows', 10)

# Display all columns
pd.set_option('display.max_columns', 20)

# Display full width of columns
pd.set_option('display.max_colwidth', None)

print("Done loading imports")

# GLOBAL VARIABLES

WBD_HUC8_FILE_PATH = f"{ROOT_PATH}/inputs/wbd/WBD_National_HUC8_EPSG_5070_HAND_domain.gpkg"
FIM_100_ROOT_DIR = f"{ROOT_PATH}/ripple/fim_100_prod_data"
STATS_ORIG_DIR = f"{FIM_100_ROOT_DIR}/download_stats"
OUTPUT_DIR = f"{FIM_100_ROOT_DIR}/aggreg_prod_data"

RAS2FIM_DATA_FILE = f"{OUTPUT_DIR}/ras2fim_v2_huc_list_w_feature_counts.csv"
OUTPUT_METRICS_FILE = f"{OUTPUT_DIR}/fim_100_metrics.csv"
OUTPUT_HECRAS_BOUNDARY_FILE = f"{OUTPUT_DIR}/FIM_100_hecras_boundaries.csv"  # HV version with has geom
OUTPUT_HYDROVIS_HECRAS_HUC_FILE = f"{OUTPUT_DIR}FIM_100_hecras_hucs.csv" # NO geom, simplied columns for HV flow processing

print("Done loading global variables")


### Part 1 : Aggregate stats csvs (has one csv per mc download)

In [6]:
print("")
download_stats_files = glob.glob(STATS_ORIG_DIR + "/download_stats_*.csv")

if len(download_stats_files) == 0:
    print("Error: No stats file available to aggregate. Program terminated.")
    sys.exit(1)

download_stats_df_list = []
for filename in download_stats_files:
    df = pd.read_csv(filename,
                     index_col=None,
                     dtype={'huc': str,
                            'num_features': np.int32,
                            'download_size_in_mib':np.int32,
                            'download_s3_dur_in_mins_perc':float,
                            'upload_s3_dur_in_mins_perc': float,
                            'total_duration': float,
                            'date_downloaded': str
                           })
    download_stats_df_list.append(df)

download_stats_df = pd.concat(download_stats_df_list, ignore_index=True)

# Round the mins and change to ints
download_stats_df['download_duration'] = download_stats_df['download_s3_dur_in_mins_perc'].round().astype(np.int32)
download_stats_df['upload_duration'] = download_stats_df['upload_s3_dur_in_mins_perc'].round().astype(np.int32)
download_stats_df['total_processing_duration'] = download_stats_df['total_duration'].round().astype(np.int32)

# Drop the three orig columns
download_stats_df.drop(['download_s3_dur_in_mins_perc','upload_s3_dur_in_mins_perc','total_duration'], axis=1, inplace=True)

# trim off seconds if there are any in the data_downloaded file.
download_stats_df['truncated_col'] = download_stats_df['date_downloaded'].str[:13]
# copy it back to the original col
download_stats_df['date_downloaded'] = download_stats_df['truncated_col']
# dump the temp trunc col
download_stats_df.drop('truncated_col', axis=1, inplace=True)

download_stats_df.rename(columns={'download_size_in_mib': 'folder_size_in_mib'}, inplace=True)

print(f"Done loading {len(download_stats_df)} files")


Done loading 980 files


In [ ]:
# print(download_stats_df)
download_stats_df.head(10)
# download_stats_df.info()

#df_sorted = download_stats_df.sort_values(by='date_downloaded', ascending=False)
# df_sorted.head()


### Part 2 : Load orig RTX list, merge with agg stats df, adjust df for data type, bad data, etc

In [7]:
## Part 2_a: Load full original RTX MC list so we can compare to stats files we just downloaded. Ones that are missing are "bad"

# This is the full list takign from rtx s3 bucket and includes all folders in their "collections" folder.
print("")
rtx_orig_mc_download_list_file = f"{OUTPUT_DIR}/rtx_fim100_s3_collection_folder_names.txt"
rtx_orig_mcs_df = pd.read_csv(rtx_orig_mc_download_list_file, header=None, names=["model_collection_name"])

print(f"Done loading {len(rtx_orig_mcs_df)} files")


Done loading 992 files


In [ ]:
rtx_orig_mcs_df.head()

In [8]:
## Part 2_b: Merge orig rtx with filtered rtx list

merged_rtx_df = pd.merge(rtx_orig_mcs_df, download_stats_df, on='model_collection_name', how='left')

# Fix data types and NaNs
# adjust the NaN
merged_rtx_df[['source', 'huc', 'date_downloaded']] = merged_rtx_df[['source', 'huc', 'date_downloaded']].fillna('')

merged_rtx_df[['num_features', 'folder_size_in_mib', 'download_duration', 'upload_duration', 
    'total_processing_duration']] = merged_rtx_df[['num_features', 'folder_size_in_mib', 'download_duration', 'upload_duration', 
    'total_processing_duration']].fillna(0)

merged_rtx_df['num_features'] = merged_rtx_df['num_features'].astype(np.int32)
merged_rtx_df['folder_size_in_mib'] = merged_rtx_df['folder_size_in_mib'].astype(np.int32)
merged_rtx_df['download_duration'] = merged_rtx_df['download_duration'].astype(np.int32)
merged_rtx_df['upload_duration'] = merged_rtx_df['upload_duration'].astype(np.int32)
merged_rtx_df['total_processing_duration'] = merged_rtx_df['total_processing_duration'].astype(np.int32)

print(f"merged_rtx loaded with {len(merged_rtx_df)} recs")


merged_rtx loaded with 992 recs


In [ ]:
merged_rtx_df.head()
 #merged_rtx_df.info()

In [9]:
## Part 2_c: Look for invalid recs (ie.. a stats csv that did not have a matching mc_name from the master list

# ---------------
# add a new column
merged_rtx_df.insert(loc=3, column='is_invalid_mc', value='')

# Add column to show if huc has both mip and ble
merged_rtx_df.insert(loc=5, column='has_mip_and_ble_mcs', value='')

print(f"merged_rtx is_valid_huc and has_mip_and_ble_mcs columns added")

merged_rtx is_valid_huc and has_mip_and_ble_mcs columns added


In [ ]:
merged_rtx_df.head()

In [10]:
for index, row in merged_rtx_df.iterrows():
    huc8 = row['huc']
    if huc8 == '' or len(huc8) != 8:
        merged_rtx_df.at[index, 'is_invalid_mc'] = 'True'
    else:
        merged_rtx_df.at[index, 'is_invalid_mc'] = ' '        
        count_matching_hucs = len(merged_rtx_df[merged_rtx_df['huc'] == huc8])
        if count_matching_hucs > 1:
            merged_rtx_df.at[index, 'has_mip_and_ble_mcs'] = 'True'
        else:
            merged_rtx_df.at[index, 'has_mip_and_ble_mcs'] = ' '

print("adjustments for merging and invalid recs complete")

adjustments for merging and invalid recs complete


In [11]:
# merged_rtx_df.head()
# merged_rtx_df.info()
#test_df = merged_rtx_df.loc[merged_rtx_df['is_invalid_mc'] == 'True']
# print(f"Number of invalid mc is {len(test_df)}")
# test_df.head(100)

test_df = merged_rtx_df.loc[merged_rtx_df['has_mip_and_ble_mcs'] == 'True']
print(len(test_df))
# test_df.head(100)

142


In [12]:
# Part 2_d : Save a version of this that can drop right into a google sheet in the order and headers we want

rtx_mc_records_col_unordered_df = pd.DataFrame(merged_rtx_df)

rtx_mc_records_col_unordered_df['notes'] = ''  # used in the google sheets

# reorder columns
new_col_order = ['model_collection_name', 'source', 'huc', 'is_invalid_mc', 'has_mip_and_ble_mcs', 'num_features',
                 'folder_size_in_mib', 'total_processing_duration', 'date_downloaded', 'notes',
                 'download_duration', 'upload_duration']
rtx_mc_records_col_ordered_df = rtx_mc_records_col_unordered_df.reindex(columns=new_col_order)

print("reordered df made")


reordered df made


In [ ]:
rtx_mc_records_col_ordered_df.head()

In [13]:
# Save it as a csv
metrics_df = rtx_mc_records_col_ordered_df.sort_values(by='huc')
metrics_df.to_csv(OUTPUT_METRICS_FILE, index=False)
print("FIM 100 Metrics file saved")

FIM 100 Metrics file saved


### Part 3 : Merge ras2fim v2 data into it. It has 12 datasets based in HUCs not MC

In [ ]:
# We aleady have a merged ras2fim v2 csv, so we don't need to reload it

# stats_files = glob.glob(RIPPLE_STATS_CSV_DIR + "*.csv")
# df_ripple_stats_files = []

# for filename in stats_files:
#     df = pd.read_csv(filename,
#                      index_col=None,
#                      usecols=['huc', 'source', 'num_features'],
#                      dtype={'huc': str})
#     df_ripple_stats_files.append(df)

# df_ripple_stats = pd.concat(df_ripple_stats_files, ignore_index=True)

# print(f"df_ripple_stats loaded with {len(df_ripple_stats)} recs")


In [14]:
# Load the ras2fim data
df_ras2fim = pd.read_csv(RAS2FIM_DATA_FILE,
                         index_col=None,
                         dtype={'huc': str})
df_ras2fim["source"] = "ras2fim"

print('ras2fim loaded')
# df_ras2fim.head(20)


ras2fim loaded


In [ ]:
df_ras2fim.head()

In [15]:
# merge the ripple df and the ras2fim df
df_stats = pd.concat([merged_rtx_df, df_ras2fim], ignore_index=True)

df_stats['huc'] = df_stats['huc'].str.zfill(8)
print("Merge complete")

Merge complete


In [ ]:
df_stats.head()

### Part 4 : Pivot to become HUC centric

In [62]:
# Group by 'Category' and pivot 'Item' to columns
df_pivot_w_index = df_stats.pivot_table(index='huc', columns='source', values='num_features')
#df_pivot = df_stats.pivot_table(index='huc', columns='source', values='num_features')

# the pivot makes a weird index, so copy the index to a temp column column then we can remove it
# df_pivot = df_pivot_w_index.reset_index().rename(columns={'source':'temp_index'})
df_pivot = df_pivot_w_index.reset_index().rename(columns={'huc':'huc8'})
# df_pivot_reset = df_pivot_w_index.reset_index()
# df_pivot = df_pivot_w_index.reset_index(drop=True)
df_pivot.drop('', axis=1, inplace=True)
# df_pivot.drop('source', axis=1, inplace=True)

df_pivot["ble"].fillna("0", inplace = True)
df_pivot["mip"].fillna("0", inplace = True)
df_pivot["ras2fim"].fillna("0", inplace = True)

df_pivot['ble'] = df_pivot['ble'].astype(int)
df_pivot['mip'] = df_pivot['mip'].astype(int)
df_pivot['ras2fim'] = df_pivot['ras2fim'].astype(int)

# drop blank rows
df_pivot = df_pivot[ ((df_pivot["ble"] > 0) | (df_pivot["mip"] > 0) | (df_pivot["ras2fim"] > 0)) ]

# df_pivot.drop('source', axis=1, inplace=True)

print(f"pivot complete. Number of recs (unique valid hucs which have at least some featues from one of the three sources) are {len(df_pivot)}")

# note: 909 is the total and matches unique hucs from our google sheet


pivot complete. Number of recs (unique valid hucs which have at least some featues from one of the three sources) are 909


In [48]:
# print(df_pivot['source'])

# df_pivot['test_index_column'] = df_pivot.index

# just testing
#df_pivot.head()
# df_pivot.info()
# print(df_pivot.index.name)
print(df_pivot.columns)
#df = df_pivot.loc[df_pivot['ble'] > 0]
# df_pivot.loc[(df_pivot['mip'] == 0) & (df_pivot['ble'] > 0) ]
# df_pivot.loc[df_pivot['ras2fim'] > 0]


Index(['huc8', 'ble', 'mip', 'ras2fim'], dtype='object', name='source')


### Part 5 : Make a HydroVIS table called hecras_boundaries for the static services, inc add geometries from wbd

In [63]:
# Part 5.a figure out which is the selected source based on whic has the most features/

# find the source with the highest number of features.
cols_to_check = ['ble', 'mip', 'ras2fim']
# df_pivot["selected_source"] = 
df_pivot["selected_source"] = df_pivot[cols_to_check].idxmax(axis=1)

print("selected source calculated")

selected source calculated


In [64]:
# table adjustments
df_pivot.rename(columns={"ble": "num_ble_features", "mip": "num_mip_features", "ras2fim": "num_ras2fim_features"}, inplace=True)

# all are active, but in HV some might be changed on the fly to false.
# Note.. not all HUCs are in here, only the ones that are applicable.
df_pivot["is_active"] = "True"

# df_pivot.to_csv(OUTPUT_WO_GEOM_CSV_PATH)
# os.chmod(OUTPUT_WO_GEOM_CSV_PATH, stat.S_IRWXU | stat.S_IRWXG | stat.S_IRWXO)

# print(f"boundaries without geom saved at {OUTPUT_WO_GEOM_CSV_PATH}")
print("columns renamed")

columns renamed


In [65]:
df_pivot.head()

source,huc8,num_ble_features,num_mip_features,num_ras2fim_features,selected_source,is_active
1,01020002,0,1,0,mip,True
2,01020003,0,36,0,mip,True
3,01020004,0,4,0,mip,True
4,01020005,0,311,0,mip,True
5,01030003,0,23,0,mip,True


In [69]:
# Part 5_b:  Add geometries from the HUCs from the WBD 

print("This is slow to execute (loading wbd)")
# Load the WBD
wbd = gpd.read_file(WBD_HUC8_FILE_PATH)[["HUC8", "geometry"]]

# merge with my csv  (gpd)
boundaries_df = df_pivot.merge(wbd, left_on='huc8', right_on='HUC8')

boundaries_df.drop('HUC8', axis=1, inplace=True)

# The implied CRS is epsg:5070
# Note.. we do not have any AK so we can leave it as 5070 for now
print(f"Total Rec count is {len(boundaries_df)}")

# 907.. huh? lost two, hummm  (outside our conus 5070 domain? - Canada or Mexico?)

This is slow to execute (loading wbd)
Total Rec count is 907


In [70]:
# boundaries_df.head(1)
boundaries_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 907 entries, 0 to 906
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   huc8                  907 non-null    object  
 1   num_ble_features      907 non-null    int64   
 2   num_mip_features      907 non-null    int64   
 3   num_ras2fim_features  907 non-null    int64   
 4   selected_source       907 non-null    object  
 5   is_active             907 non-null    object  
 6   geometry              907 non-null    geometry
dtypes: geometry(1), int64(3), object(3)
memory usage: 56.7+ KB


In [71]:

# Save the boundaries file (with geom)
boundaries_df.to_csv(OUTPUT_HECRAS_BOUNDARY_FILE, index=False)
os.chmod(OUTPUT_HECRAS_BOUNDARY_FILE, stat.S_IRWXU | stat.S_IRWXG | stat.S_IRWXO)

print(f"Hecras boundaries file with geometries saved")


Hecras boundaries file with geometries saved


In [ ]:
# Part 5_c:  The hecras_hucs list file
# This is a simplified file without geometry and is used by HydroVIS for runtime flow processing

hecras_hucs_df = boundaries_df


In [ ]:
#OUTPUT_HYDROVIS_HECRAS_HUC_FILE = f"{OUTPUT_DIR}FIM_100_hecras_hucs.csv" # NO geom, simplied columns for HV flow processing